In [2]:
# Импорт необходимых библиотек

import numpy as np
import random as rnd
import sys
# import matplotlib.pyplot as plt  #Пока не надо

In [ ]:
# Входные данные

# Параметры откачиваемого газа

gas_constant = 8.31446262 # k - постоянная больцмана
temperature = 293 # T - в Кельвинах
molar_mass = 0.028 # M - молярная масса *(воздуха)
Uh = 1000 * (2 * gas_constant * temperature / molar_mass)**0.5 # Наиболее вероятная скорость (мм/с)

# Параметры колеса

radius = 70 # r
radius_periphery = 100 # R

# Скорость вращения колеса

speed_periphery = Uh * 0.6 # V.p [мм/с] 
speed_center = speed_periphery * radius / radius_periphery # V.c [мм/с]

omega_center = speed_center / radius # [1/с]

# Геометрия канала

total_width = 3 # c * sin (psi)  

channel_height = 1 # p
channel_rotation = float(np.pi/6) # psi

# Отношение a/с
a_c = 1


blade_width = float(total_width / np.sin(channel_rotation)) # c 
blade_height = radius_periphery - radius # b (по оси лопатки)
blade_interval = a_c * blade_width # a
blade_depth = 0.5 # g тольщина вершины лопатки
blade_depth_root = 1.5 # g толщина у корня лопатки
blade_angle_center = float(np.arcsin( (blade_interval + blade_depth_root) / (2 * radius))) #fi оси лопатки
blade_angle_naklona = float(np.arctan( (blade_depth_root - 2 * blade_depth) / blade_height )) # Наклон боковины лопатки относительно своего центра
blade_angle = float(np.pi/2) - blade_angle_center - blade_angle_naklona # суммарный угол наклона от оси X до боковой поверхности лопатки

# Проекции

total_height = float(blade_height * np.sin(blade_angle)) # h
upper = np.tan(blade_angle_center) * (total_height + radius)

projection_total_height = float(blade_height * np.cos(blade_angle)) # b*cos(fi)
projection_total_width = float(blade_width * np.cos(channel_rotation)) # c * cos (psi)
print(projection_total_width)
# Вспомогательные углы

fi0 = blade_angle_center # угол центра лопатки
fi1 = float(np.arcsin( blade_interval / (2 * radius) )) # угол для А и B
fi2 = float(np.arctan( (upper - blade_depth/2) / (total_height + radius) )) # угол для C, D
fi3 = float(np.arctan( (upper + blade_depth/2) / (total_height + radius) )) # угол для G, H
fi4 = float(np.arctan( (upper + blade_depth/2) / (total_height + radius + channel_height) )) # угол для E, F



5.196152422706633


In [4]:
# Центр системы координат

thetta = 0 # угол поворота центра СК

CENTER_AXIS = [0, 0, 0]
CENTER_CIRCLE = [0, 0, -1*radius]

points_array = [[], [], [], [], [], [], [], [], [], []]
surfaces = [[], [], [], [], [], [], [], [], [], []]

In [5]:
# Функция движения центра координатных осей

def turning_center(CENTER_AXIS, dt, omega_center, thetta, radius):
    
    y = CENTER_AXIS[1]
    
    # Обновляем угол тетта
    thetta += omega_center * dt
    
    # Вычисляем локальные координаты (относительно центра окружности)
    x_local = radius * np.sin(thetta)
    z_local = radius * np.cos(thetta)
    
    # Переход к глобальной системе координат
    x_global = x_local
    z_global = z_local - radius
    
    centr_axs = [float(x_global), float(y), float(z_global)]
    
    
    return(centr_axs, thetta)
    

In [6]:
# Нахождение опорных точек поверхностей

def get_surface_point(channel_height, thetta, points_array):
    
    
    # Точки для пов-ти входа
    A0 = [float(np.sin(thetta - fi1) * (radius/np.cos(fi1))),
          float(0),
          float(np.cos(thetta - fi1) * (radius/np.cos(fi1))) - radius]
    
    B0 = [float(np.sin(thetta + fi1) * (radius/np.cos(fi1))),
          float(0),
          float(np.cos(thetta + fi1) * (radius/np.cos(fi1))) - radius]
    
    D0 = [float(np.sin(thetta - fi2) * (radius_periphery/(np.cos(fi0)*np.cos(fi2)))),
          float(0),
          float(np.cos(thetta - fi2) * (radius_periphery/(np.cos(fi0)*np.cos(fi2)))) - radius]
    
    C0 = [float(np.sin(thetta + fi2) * (radius_periphery/(np.cos(fi0)*np.cos(fi2)))),
          float(0),
          float(np.cos(thetta + fi2) * (radius_periphery/(np.cos(fi0)*np.cos(fi2)))) - radius]
    
    G0 = [float(np.sin(thetta - fi3) * (radius_periphery/(np.cos(fi0)*np.cos(fi3)))),
          float(0),
          float(np.cos(thetta - fi3) * (radius_periphery/(np.cos(fi0)*np.cos(fi3)))) - radius]
    
    H0 = [float(np.sin(thetta + fi3) * (radius_periphery/(np.cos(fi0)*np.cos(fi3)))),
          float(0),
          float(np.cos(thetta + fi3) * (radius_periphery/(np.cos(fi0)*np.cos(fi3)))) - radius]
    
    F0 = [float(np.sin(thetta - fi4) * ( (radius_periphery/(np.cos(fi0)) + channel_height) / np.cos(fi4) )),
          float(0),
          float(np.cos(thetta - fi4) * ( (radius_periphery/(np.cos(fi0)) + channel_height) / np.cos(fi4) )) - radius]
    
    E0 = [float(np.sin(thetta + fi4) * ( (radius_periphery/(np.cos(fi0)) + channel_height) / np.cos(fi4) )),
          float(0),
          float(np.cos(thetta + fi4) * ( (radius_periphery/(np.cos(fi0)) + channel_height) / np.cos(fi4) )) - radius]
    
    points_array[0] = [A0, B0, C0, D0, E0, F0, G0, H0]

    #Точки для пов-ти выхода
    A1 = [float(A0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(A0[2] + projection_total_width*np.sin(thetta))]
    
    B1 = [float(B0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(B0[2] + projection_total_width*np.sin(thetta))]
    
    C1 = [float(C0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(C0[2] + projection_total_width*np.sin(thetta))]
    
    H1 = [float(H0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(H0[2] + projection_total_width*np.sin(thetta))]
    
    E1 = [float(E0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(E0[2] + projection_total_width*np.sin(thetta))]
    
    F1 = [float(F0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(F0[2] + projection_total_width*np.sin(thetta))]
    
    G1 = [float(G0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(G0[2] + projection_total_width*np.sin(thetta))]
    
    D1 = [float(D0[0] - projection_total_width*np.cos(thetta)),
          float(total_width),
          float(D0[2] + projection_total_width*np.sin(thetta))]
    
    points_array[1] = [A1, B1, C1, D1, E1, F1, G1, H1]
    
    #Точки для пов-ти левой лопатки
    A2 = A0
    B2 = A1
    C2 = D1
    D2 = D0
    
    points_array[2] = [A2, B2, C2, D2]

    #Точки для пов-ти лев. зазора
    A3 = G0
    B3 = G1
    C3 = F1
    D3 = F0
    
    points_array[3] = [A3, B3, C3, D3]

    #Точки для пов-ти прав. лопатки
    A4 = B0
    B4 = B1
    C4 = C1
    D4 = C0
    
    points_array[4] = [A4, B4, C4, D4]

    #Точки для пов-ти прав. зазора
    A5 = H0
    B5 = H1
    C5 = E1
    D5 = E0
    
    points_array[5] = [A5, B5, C5, D5]

    #Точки для пов-ти верха
    A6 = F0
    B6 = E0
    C6 = E1
    D6 = F1
    
    points_array[6] = [A6, B6, C6, D6]

    #Точки для пов-ти низа
    A7 = A0
    B7 = B0
    C7 = B1
    D7 = A1
    
    points_array[7] = [A7, B7, C7, D7]

    #Точки для пов-ти верх лев.лопатки
    A8 = G0
    B8 = D0
    C8 = D1
    D8 = G1
    
    points_array[8] = [A8, B8, C8, D8]

    #Точки для пов-ти верх прав.лопатки
    A9 = C0
    B9 = H0
    C9 = H1
    D9 = C1
    
    points_array[9] = [A9, B9, C9, D9]
    
    
    return(points_array)

In [7]:
# Точки для поверхностей, через входные данные и центр

def get_surf_point(channel_height, CENTER_AXIS,
                   tangent_angle, points_array):
       
    x = CENTER_AXIS[0]   
    y = CENTER_AXIS[1]   
    z = CENTER_AXIS[2]
       
    # Точки для пов-ти входа   
    A0 = [float(x - ((blade_interval/2)*np.cos(tangent_angle))),
          float(y),
          float(z + (np.sin(tangent_angle)*(blade_interval/2)))]
    
    B0 = [float(x + ((blade_interval/2)*np.cos(tangent_angle))),
          float(y),
          float(z - (np.sin(tangent_angle)*(blade_interval/2)))]
    
    C0 = [float(x + ((blade_interval/2)*np.cos(tangent_angle) + blade_height*np.cos(blade_angle - tangent_angle))),
          float(y),
          float(z + blade_height*np.sin(blade_angle - tangent_angle) - (blade_interval/2)*np.sin(tangent_angle))]
    
    H0 = [float(x + ((blade_interval/2)*np.cos(tangent_angle) + blade_height*np.cos(blade_angle - tangent_angle)) + blade_depth*np.cos(tangent_angle)),
          float(y),
          float(z + blade_height*np.sin(blade_angle - tangent_angle - (blade_interval/2)*np.sin(tangent_angle)) - blade_depth*np.sin(tangent_angle))]
    
    E0 = [float(x + ((blade_interval/2)*np.cos(tangent_angle) + blade_height*np.cos(blade_angle - tangent_angle)) + blade_depth*np.cos(tangent_angle) + channel_height*np.sin(tangent_angle)),
          float(y),
          float(z + blade_height*np.sin(blade_angle - tangent_angle - (blade_interval/2)*np.sin(tangent_angle)) - blade_depth*np.sin(tangent_angle) + channel_height*np.cos(tangent_angle))]
    
    F0 = [float(x - (blade_height*np.cos(blade_angle - tangent_angle) + (blade_interval/2)*np.cos(tangent_angle)) - blade_depth*np.cos(tangent_angle) + channel_height*np.sin(tangent_angle)),
          float(y),
          float(z + (blade_height*np.sin(blade_angle - tangent_angle) + (blade_interval/2)*np.sin(tangent_angle)) + blade_depth*np.sin(tangent_angle) + channel_height*np.cos(tangent_angle))]
    
    G0 = [float(x - (blade_height*np.cos(blade_angle - tangent_angle) + (blade_interval/2)*np.cos(tangent_angle)) - blade_depth*np.cos(tangent_angle)),
          float(y),
          float(z + (blade_height*np.sin(blade_angle - tangent_angle) + (blade_interval/2)*np.sin(tangent_angle)) + blade_depth*np.sin(tangent_angle))]
    
    D0 = [float(x - blade_height*np.cos(blade_angle - tangent_angle) - blade_interval/2*np.cos(tangent_angle)),
          float(y),
          float(z + (blade_height*np.sin(blade_angle - tangent_angle) + (blade_interval/2)*np.sin(tangent_angle)))]
    
    points_array[0] = [A0, B0, C0, D0, E0, F0, G0, H0]

    #Точки для пов-ти выхода
    A1 = [float(A0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(A0[2] + projection_total_width*np.sin(tangent_angle))]
    
    B1 = [float(B0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(B0[2] + projection_total_width*np.sin(tangent_angle))]
    
    C1 = [float(C0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(C0[2] + projection_total_width*np.sin(tangent_angle))]
    
    H1 = [float(H0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(H0[2] + projection_total_width*np.sin(tangent_angle))]
    
    E1 = [float(E0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(E0[2] + projection_total_width*np.sin(tangent_angle))]
    
    F1 = [float(F0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(F0[2] + projection_total_width*np.sin(tangent_angle))]
    
    G1 = [float(G0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(G0[2] + projection_total_width*np.sin(tangent_angle))]
    
    D1 = [float(D0[0] - projection_total_width*np.cos(tangent_angle)),
          float(y + total_width),
          float(D0[2] + projection_total_width*np.sin(tangent_angle))]
    
    points_array[1] = [A1, B1, C1, D1, E1, F1, G1, H1]
    
    #Точки для пов-ти левой лопатки
    A2 = A0
    B2 = A1
    C2 = D1
    D2 = D0
    
    points_array[2] = [A2, B2, C2, D2]

    #Точки для пов-ти лев. зазора
    A3 = G0
    B3 = G1
    C3 = F1
    D3 = F0
    
    points_array[3] = [A3, B3, C3, D3]

    #Точки для пов-ти прав. лопатки
    A4 = B0
    B4 = B1
    C4 = C1
    D4 = C0
    
    points_array[4] = [A4, B4, C4, D4]

    #Точки для пов-ти прав. зазора
    A5 = H0
    B5 = H1
    C5 = E1
    D5 = E0
    
    points_array[5] = [A5, B5, C5, D5]

    #Точки для пов-ти верха
    A6 = F0
    B6 = E0
    C6 = E1
    D6 = F1
    
    points_array[6] = [A6, B6, C6, D6]

    #Точки для пов-ти низа
    A7 = A0
    B7 = B0
    C7 = B1
    D7 = A1
    
    points_array[7] = [A7, B7, C7, D7]

    #Точки для пов-ти верх лев.лопатки
    A8 = G0
    B8 = D0
    C8 = D1
    D8 = G1
    
    points_array[8] = [A8, B8, C8, D8]

    #Точки для пов-ти верх прав.лопатки
    A9 = C0
    B9 = H0
    C9 = H1
    D9 = C1
    
    points_array[9] = [A9, B9, C9, D9]
    
    
    return(points_array)

In [8]:
# Функция вычисления коэффициентов A, B, C, D плоскостей

def get_coefficient(T1, T2, T3):
    x = 0
    y = 1
    z = 2
    
    A = T1[y]*(T2[z]-T3[z]) + T2[y]*(T3[z]-T1[z]) + T3[y]*(T1[z]-T2[z])
    B = T1[z]*(T2[x]-T3[x]) + T2[z]*(T3[x]-T1[x]) + T3[z]*(T1[x]-T2[x])
    C = T1[x]*(T2[y]-T3[y]) + T2[x]*(T3[y]-T1[y]) + T3[x]*(T1[y]-T2[y])
    D = - T1[x]*(T2[y]*T3[z] - T3[y]*T2[z]) - T2[x]*(T3[y]*T1[z] - T1[y]*T3[z]) - T3[x]*(T1[y]*T2[z] - T2[y]*T1[z])
    
    n = (A**2 + B**2 + C**2)**0.5 # Длина вектора нормали
    
    if n != 1:
        # Нормировка коэффициентов
        A = A/n
        B = B/n
        C = C/n
        D = D/n
    
    
    return(A,B,C,D)

In [9]:
# Уравнения поверхностей в общем виде

def get_surfaces(points_array):
    
    inlet = get_coefficient(points_array[0][0], points_array[0][1], points_array[0][2]) #0

    outlet = get_coefficient(points_array[1][0], points_array[1][1], points_array[1][2]) #1

    left_blade = get_coefficient(points_array[2][0], points_array[2][1], points_array[2][2]) #2

    left_channel = get_coefficient(points_array[3][0], points_array[3][1], points_array[3][2]) #3 -> 8

    right_blade = get_coefficient(points_array[4][0], points_array[4][1], points_array[4][2]) #4

    right_channel = get_coefficient(points_array[5][0], points_array[5][1], points_array[5][2]) #5 -> 9

    top = get_coefficient(points_array[6][0], points_array[6][1], points_array[6][2]) #6

    bottom = get_coefficient(points_array[7][0], points_array[7][1], points_array[7][2]) #7

    left_blade_top = get_coefficient(points_array[8][0], points_array[8][1], points_array[8][2]) #8 -> 3

    right_blade_top = get_coefficient(points_array[9][0], points_array[9][1], points_array[9][2]) #9 -> 5
    
    surfaces = [inlet, outlet, left_blade, left_channel, right_blade, right_channel, top, bottom, left_blade_top, right_blade_top]
    #surfaces.index(название поверхности) - чтобы узнать индекс для обращения

    return(surfaces)

In [ ]:
# Начальная точка

def get_start_point():
    
    y = 0
    z = float((rnd.random())**0.5 * (total_height + channel_height)) # добавить/убрать channel_height
    x = float(rnd.random() * (2*z / np.tan(blade_angle) + blade_interval) - (z/np.tan(blade_angle) + blade_interval/2))
    point_start = [x, y, z]
    
    return(point_start)

In [122]:
# Начальная точка для обратного течения

def get_back_start_point():
    
    y = total_width
    z = float((rnd.random())**0.5 * (total_height + channel_height))
    x = float( (rnd.random()*((2*z / np.tan(blade_angle) + blade_interval)) - (z/np.tan(blade_angle) + blade_interval/2) - projection_total_width))
    
    point_start = [x, y, z]
    
    return(point_start)

In [12]:
# Направляющий вектор

def get_start_vector ():
    
    ksi = rnd.random()
    if ksi == 1:
        ksi = rnd.random()
    fi = 2 * np.pi * ksi
    tetta = float(np.arcsin(ksi**0.5))
    
    # Поиск направляющих косинусов (cos_alpha -> x, cos_betta -> y, cos_gamma -> z)
    cos_alpha = float(np.sin(tetta) * np.cos(fi))
    cos_betta = float(np.cos(tetta))
    cos_gamma = float(np.sin(tetta) * np.sin(fi))
        
    vector_cos = [cos_alpha, cos_betta, cos_gamma]
    
    return(vector_cos)

In [ ]:
# Направляющий вектор, для обратного потока

def get_back_start_vector():
    
    ksi = rnd.random()
    if ksi == 1:
        ksi = rnd.random()
    fi = 2 * np.pi * ksi
    tetta = float(np.arcsin(ksi**0.5))
    
    # Поиск направляющих косинусов (cos_alpha -> x, cos_betta -> y, cos_gamma -> z)
    cos_alpha = -1*float(np.sin(tetta) * np.cos(fi))
    cos_betta = -1*float(np.cos(tetta))
    cos_gamma = -1*float(np.sin(tetta) * np.sin(fi))
        
    vector_cos = [cos_alpha, cos_betta, cos_gamma]
    
    return(vector_cos)

In [14]:
# Направляющий вектор в пересчете от другой нормали

def get_new_start_vector(points_array, vector_cos, surfaces, index):
    
    # Первый нормированный вектор
    if index == 2 or index == 4 or index == 6 or index == 8:
        X_new = (points_array[0][index-2][0]-points_array[1][index-2][0],
                 points_array[0][index-2][1]-points_array[1][index-2][1],
                 points_array[0][index-2][2]-points_array[1][index-2][2])
        X_new = X_new/ np.linalg.norm(X_new)
    
    if index == 7 or index == 9:
        X_new = (points_array[0][index-7][0]-points_array[1][index-7][0],
                 points_array[0][index-7][1]-points_array[1][index-7][1],
                 points_array[0][index-7][2]-points_array[1][index-7][2])    
        X_new = X_new/ np.linalg.norm(X_new)
        
    # Второй нормированный вектор (нормаль)    
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    
    Y_new = (A, B, C)
    
    # Третий нормированный вектор

    Z_new = np.cross(Y_new, X_new)
    Z_new = Z_new/ np.linalg.norm(Z_new)
    
    # Матрица поворота
    
    R = np.column_stack((X_new, Y_new, Z_new))
    
    # Генерация вектора в X'Y'Z' 
    
    ksi = rnd.random()
    if ksi == 1:
        ksi = rnd.random()
    fi = 2 * np.pi * ksi
    tetta = float(np.arcsin(ksi**0.5))
    
    new_start_vec = np.array([float(np.sin(tetta)*np.cos(fi)),
                             float(np.cos(tetta)),
                             float(np.sin(tetta)*np.sin(fi))])
    
    # Переход в исходную СК
    
    vector_cos = R @ new_start_vec
    
    if index == 4 or index == 6:
        vector_cos = -1 * vector_cos
        
    return(vector_cos)

In [ ]:
# Координаты точки при движении

def get_new_point (point_start, vector_cos, dt, molecule_speed):
    
    V = molecule_speed
    
    x = point_start[0]
    y = point_start[1]
    z = point_start[2]
    
    cosa = vector_cos[0]
    cosb = vector_cos[1]
    cosg = vector_cos[2]
    
    # На всякий случай оставить расчет параметра здесь
    # t = -1 * (A*x + B*y + C*z + D) / (A*cosa + B*cosb + C*cosg)
    
    x_new = cosa * V *dt + x
    y_new = cosb * V *dt + y
    z_new = cosg * V *dt + z
    
    new_point = [x_new, y_new, z_new]
    return(new_point)

In [16]:
# Функция для определения положения точки относительно прямой (для метода треугольников)

def sign(p1, p2, p3):
    
    #Погрешность
    delta = 0.000000001
    
    value = (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])
    if abs(value) < delta:
        return 0  # Считаем точку лежащей на прямой
    return value

In [17]:
# Подфункция для проверки на принадлежность треугольнику

def point_in_triangle(point, p1, p2, p3):
    
    # Погрешность
    delta = 0.000000001
    
    # Проверяем расположение точки относительно сторон, определя знак + или -
    d1 = sign(point, p1, p2)
    d2 = sign(point, p2, p3)
    d3 = sign(point, p3, p1)
    
    # Проверяем есть ли хотя бы один - и есть ли хотя бы один +
    negativ_sign = (d1 < -delta) or (d2 < -delta) or (d3 < -delta)
    positiv_sign = (d1 > delta) or (d2 > delta) or (d3 > delta)
    
    # Если есть + и - одновременно, то точка находится вне треугольника, поэтому выводим [not (neg and pos)]
    return not (negativ_sign and positiv_sign)

In [18]:
# Функция получения расстояния от точки до плоскости

def get_distance(point, index, surfaces):
    
    #Проверка на принаджлежность к плоскости через подстановку в каноническое уравнение
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    D = surfaces[index][3]
    
    square = (A*point[0] + B*point[1] + C*point[2] + D) / ((A**2 + B**2 + C**2)**(0.5))
    
    
    return(square)

In [18]:
# Функция проецирования предыдушей и текущей точек на плоскость столкновения...
# ... и нахождение средней точки на прямой, образованной двумя проекциями

def middle_projection(index, surfaces, prev_point, distance_prev, curr_point, distance_curr):
    
    #Коэффициенты проверяемой плоскости
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    D = surfaces[index][3]
    
    #Спроецируем точки на плоскость, для нахождения серединной точки между ними
        #Нормаль и её длина
    norm = np.array ([A, B, C])
    norm_len = np.linalg.norm(norm)
    n_normalized = norm / norm_len
    
        #Проекции точек на плоскость
    proj_prev = np.array(prev_point) - distance_prev[index] * n_normalized
    proj_curr = np.array(curr_point) - distance_curr[index] * n_normalized
    
        #Поиск средней точки на прямой, образованной двумя спроецированными точками (попробовать уточнить положение столкновения)
    #length = np.linalg.norm(proj_curr - proj_prev)
    proj_mid = (np.array(proj_prev) + np.array(proj_curr)) / 2 # * (distance_prev[index]/distance_curr[index]))
    
    
    return(proj_mid)

In [19]:
# Проверка по площадям

def area(index, points_array, proj_point):
    
    #Тестер
    flag = False
    eps = 0.000001
    
    #Индекс поверхности
    k = index
    
    #Точка
    px = np.array([proj_point[0], proj_point[1], proj_point[2]])
    
    #Точки поверхностей
    p0 = np.array([points_array[k][0][0],points_array[k][0][1],points_array[k][0][2]])
    p1 = np.array([points_array[k][1][0],points_array[k][1][1],points_array[k][1][2]])
    p2 = np.array([points_array[k][2][0],points_array[k][2][1],points_array[k][2][2]])
    p3 = np.array([points_array[k][3][0],points_array[k][3][1],points_array[k][3][2]])
    #Дополнительные точки поверхностей входа и выхода
    if k == 1 or k == 0:
        p4 = np.array([points_array[k][4][0],points_array[k][4][1],points_array[k][4][2]])
        p5 = np.array([points_array[k][5][0],points_array[k][5][1],points_array[k][5][2]])
        p6 = np.array([points_array[k][6][0],points_array[k][6][1],points_array[k][6][2]])
        p7 = np.array([points_array[k][7][0],points_array[k][7][1],points_array[k][7][2]])
    
    #Вычисление необходимых площадей
    p1p0 = p1 - p0
    p1p2 = p1 - p2
    cross_product = np.cross(p1p0, p1p2)
    area_012 = 0.5 * np.linalg.norm(cross_product)
    
    p3p0 = p3 - p0
    p3p2 = p3 - p2
    cross_product = np.cross(p3p0, p3p2)
    area_230 = 0.5 * np.linalg.norm(cross_product)
    
    sum_area_0123 = area_012 + area_230
    
    pxp0 = px - p0
    pxp1 = px - p1
    pxp2 = px - p2
    pxp3 = px - p3
    
    cross_product = np.cross(pxp0, pxp1)
    area_x01 = 0.5 * np.linalg.norm(cross_product)

    cross_product = np.cross(pxp1, pxp2)
    area_x12 = 0.5 * np.linalg.norm(cross_product)
    
    cross_product = np.cross(pxp2, pxp3)
    area_x23 = 0.5 * np.linalg.norm(cross_product)
    
    cross_product = np.cross(pxp3, pxp0)
    area_x30 = 0.5 * np.linalg.norm(cross_product)
    
    sum_p1 = area_x01 + area_x12 + area_x23 + area_x30
    
    if abs(sum_area_0123 - sum_p1) <= eps:
        flag = True
        return (flag)
    
    if k == 1 or k == 0:
        
        p5p4 = p5 - p4
        p5p6 = p5 - p6
        cross_product = np.cross(p5p4, p5p6)
        area_456 = 0.5 * np.linalg.norm(cross_product)
    
        p7p4 = p7 - p4
        p7p6 = p7 - p6
        cross_product = np.cross(p7p4, p7p6)
        area_674 = 0.5 * np.linalg.norm(cross_product)
    
        sum_area_4567 = area_456 + area_674
        
        pxp4 = px - p4
        pxp5 = px - p5
        pxp6 = px - p6
        pxp7 = px - p7
    
        cross_product = np.cross(pxp4, pxp5)
        area_x45 = 0.5 * np.linalg.norm(cross_product)

        cross_product = np.cross(pxp5, pxp6)
        area_x56 = 0.5 * np.linalg.norm(cross_product)
    
        cross_product = np.cross(pxp6, pxp7)
        area_x67 = 0.5 * np.linalg.norm(cross_product)
    
        cross_product = np.cross(pxp7, pxp4)
        area_x74 = 0.5 * np.linalg.norm(cross_product)
    
        sum_p2 = area_x45 + area_x56 + area_x67 + area_x74
        
        if abs(sum_area_4567 - sum_p2) <= eps:
            flag = True
            return (flag)
    
    return(flag)
        

In [20]:
# Проекция на плоскость (при условии близости точки)

def projection(index, surfaces, point):
    
    #Коэффициенты проверяемой плоскости
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    D = surfaces[index][3]
    
    #Данные точки
    x0 = point[0]
    y0 = point[1]
    z0 = point[2]
    
    numerator = A*x0 + B*y0 + C*z0 + D
    denominator = A**2 + B**2 + C**2
    t = -numerator / denominator
    
    # Вычисляем проекцию
    projct = [x0 + A*t, y0 + B*t, z0 + C*t]
    
    return(projct)
    

In [61]:
# Функция грубой проверки 

def hardcheck(index, surfaces, curr_point):
    #Заданная погрешность
    eps = 0.01
    
    #Коэффициенты проверяемой плоскости
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    D = surfaces[index][3]
    
    #Данные точки
    x = curr_point[0]
    y = curr_point[1]
    z = curr_point[2]
    
    #Проверка
    flag = False
    j = abs(A*x+B*y+C*z+D)
    
    if j <= eps:
        flag = True
        return(flag, j)
    
    return(flag, j)
    

In [22]:
# Функция проверки

def check(points_array, index, proj_mid):
    
    #Точки для поверхностей (кроме входа и выхода - у них есть дополнительные точки)
    point0 = np.array(points_array[index][0])
    point1 = np.array(points_array[index][1])
    point2 = np.array(points_array[index][2])
    point3 = np.array(points_array[index][3])
    
    #Дополнительные точки для поверхностей входа и выхода
    if index == 0 or index == 1:
        point4 = np.array(points_array[index][4])
        point5 = np.array(points_array[index][5])
        point6 = np.array(points_array[index][6])
        point7 = np.array(points_array[index][7])
    
    
    #Проверка на принадлежность треугольникам
    
    #Проверка для всех поверхностей кроме входа и выхода
    cond1 = point_in_triangle(proj_mid, point0, point1, point2)
    cond2 = point_in_triangle(proj_mid, point0, point2, point3)
    if index > 1:
        return (cond1) or (cond2)
    
    #Проверка для входа и выхода
    cond3 = point_in_triangle(proj_mid, point4, point6, point7)
    cond4 = point_in_triangle(proj_mid, point4, point5, point6)
    if index < 2:
        return (cond1) or (cond2) or (cond3) or (cond4)

In [23]:
# Нахождение какой-то координаты в начальной СК для точки после столкновения

def local_collision_point(new_point, surfaces, index, C_CIRC, C_AXIS):
    
    # Для всех индексов, кроме 6-9
    if index != 6 and index != 7 and index != 8 and index != 9:
    
        Mx = new_point[0]
        My = new_point[1]
        Mz = new_point[2]
    
        A = surfaces[7][0]
        B = surfaces[7][1]
        C = surfaces[7][2]
        D = surfaces[7][3]
    
        distance = abs(A*Mx + B*My + C*Mz + D) / ((A**2 + B**2 + C**2)**(0.5))
    
        return(distance)
    
    # Для остальных индексов
    else:
        
        Mx = new_point[0]
        My = new_point[1]
        Mz = new_point[2]
        
        # Рассчитываем прямую симметрии повернутую (oZ)
        
        vec_AB = (C_AXIS[0] - C_CIRC[0], C_AXIS[2] - C_CIRC[2])
        vec_AP = (Mx - C_CIRC[0], Mz - C_CIRC[2])
        
        cross = vec_AB[0] * vec_AP[1] - vec_AB[1] * vec_AP[0]
        
        len_AB = (vec_AB[0]**2 + vec_AB[1]**2)**0.5
        
        distance = cross/len_AB # Должно учитывать знак
        
        return(distance)

In [24]:
# Расчет новой точки отправления молекулы

def get_collision_point(index, surfaces, new_point, distance):
    
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    D = surfaces[index][3]
    
    # Для всех индексов, кроме 6-9
    if index != 6 and index != 7 and index != 8 and index != 9:
    
        y = new_point[1]
        z = distance
        x = -(B*y + C*z + D)/A
    
        return(x,y,z)
    
    # Для остальных индексов
    else:
        
        y = new_point[1]
        x = distance
        z = -(A*x + B*y + D)/C
        
        return(x,y,z)

In [25]:
# Перенос точки старта по свойству симметричности системы

def get_simmetric_point(intersection_point, surfaces, index):
    
    My = intersection_point[1]
    Mz = intersection_point[2]
    
    if index == 3:
        A = surfaces[index+2][0]
        B = surfaces[index+2][1]
        C = surfaces[index+2][2]
        D = surfaces[index+2][3]
        
        Mx = -(B*My + C*Mz + D)/A
        simmetric_point = [Mx, My, Mz]
        return(simmetric_point)
    else:
        A = surfaces[index-2][0]
        B = surfaces[index-2][1]
        C = surfaces[index-2][2]
        D = surfaces[index-2][3]
        
        Mx = -(B*My + C*Mz + D)/A
        simmetric_point = [Mx, My, Mz]
        return(simmetric_point)

In [26]:
# Разыгрывание скорости молекулы

def get_molecule_speed(Uh):
    molecule_speed = float( Uh * np.pi/7 * (7/3 + np.tan(10/13 * np.pi * (rnd.random() - 117/250))) ) # [мм/с]
    return(molecule_speed)

In [27]:
# Расчет модуля вектора скорости лопатки в точке столкновения

def get_blade_point_speed(intersection_point, speed_periphery,
                          radius, radius_periphery, CENTER_CIRCLE):
    Mx = intersection_point[0]
    Mz = intersection_point[2]
    Cx = CENTER_CIRCLE[0]
    Cz = CENTER_CIRCLE[2]
    length = ((Mx - Cx)**2 + (Mz - Cz)**2)**0.5
    blade_point_speed = (speed_periphery * (radius + length)) / (radius + radius_periphery)
    
    return(blade_point_speed)

In [28]:
# Функция сложения/вычитания векторов

def add_vectors(molecule_vector, molecule_speed,
                surfaces, index, blade_point_speed):
    
    #Вектор скорости от лопатки
    A = surfaces[index][0]
    B = surfaces[index][1]
    C = surfaces[index][2]
    magnitude = blade_point_speed
    vec1 = np.array ([magnitude*A,
                      magnitude*B,
                      magnitude*C])
    
    #Новый вектор для отраженной молекулы
    vec2 = molecule_vector * molecule_speed
    
    #Выполняем операцию
    v_res = vec1 + vec2
    new_speed = np.linalg.norm(v_res)
    new_vector = v_res / new_speed
    
    return(new_vector, new_speed)

In [ ]:
# Переосмысление

collide_index = -1
small_count_collide = 0

# Для записи результатов
N_plus=[0,0,0,0,0,0,0,0,0]
N_minus=[0,0,0,0,0,0,0,0,0]
N_propal=[0,0,0,0,0,0,0,0,0]
V2_Uh=[0,0,0,0,0,0,0,0,0]

# Исследование зависимости проводимости от величины межлопаточного зазора
for spd in range(1):
    
    speed_periphery = Uh * 0.2 * 3
    speed_center = speed_periphery * radius / radius_periphery
    omega_center = speed_center / radius
    
    # Переменные для проверки и debug'a
    outcome = 0 # ушедшие на выход
    backflow = 0 # ушедшие обратно на вход
    collide = 0 # количество столкнувшихся
    
    # Прогон молекул (в идеале 100K)
    for n in range (5000):
        
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        print(start_point)
        print(start_vector)
        
        # Костыль для обновления точки в цикли по времени
        curr_point = start_point

        # Индекс поверхности входа (0-прямой поток, 1-обратный поток)
        current_index = 0
        
        # Задаем центр СК (расположен в середине низа поверхности входа) и добавляем счетчик времени dt и угол поворота thetta
        CENTER_AXIS = [0, 0, 0]
        thetta = 0
        dt = 0
        
        # Рассчитаем начальные точки поверхностей для их задания
        CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
        points_array = get_surface_point(channel_height, thetta, points_array)
        
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
    
        
        # Цикл определяющий количество доступных отражений для молекулы (пусть пока будет 5)
        a = 5 # Переменная для количества возможных столкновений
        for count_col in range(a):
            
            
            # Создаем цикл для изменения времени (сейчас dt=10^-9)
            for time in range(3544):
                
                dt = time * 0.0000000001
            
                # Перемещаем центр СК и рассчитываем новые поверхности
                CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
                points_array = get_surface_point(channel_height, thetta, points_array)
                for j in range(len(surfaces)):
                    surfaces = get_surfaces(points_array)
            
                # Смещаем молекулу (Запоминаем прошлое на всякий)
                prev_point = curr_point
                curr_point = get_new_point(prev_point, start_vector, dt, molecule_speed)
            
                # test1 - попала в плоскость, test2 - попала в поверхность
                test1 = False
                test2 = False
                
                # Проверяем наличие попадания молекулы в поверхности
                for index in range(len(surfaces)):
                    
                    # Условие различия стартовой и проверяемой поверхностей
                    if index == current_index:
                        continue
                        
                    
                    # Проверяем попадание в плоскость
                    test1, rastoyaniye = hardcheck(index, surfaces, curr_point)
                    #if index == 1:
                        #print("расстояние до", index, 'равно', rastoyaniye)
                    if test1 == False:
                        continue
                    
                    # Проверяем на принадлежность поверхности
                    proj_point = projection(index, surfaces, curr_point)
                    test2 = area(index, points_array, proj_point)
                    if test2 == False:
                        continue
                    
                        
                    # Если test2 = true, то точка столкнулась с поверхностью
                    if test2 == True:
                        collide_index = index
                        break
                    
                       
                # Если молекула пересекла диапазон по Y с прибавкой в 20%, то выходим с цикла, получаем ошибку
                if (curr_point[1] > total_width*1.5 or curr_point[1] < -0.5*total_width):
                    break
                        
                # Если молекула улетела на выход выходим из всех циклов (времени и столкновений)
                if test2 == True and collide_index == 1:
                    break
                
                # Если молекула улетела на вход выходим из всех циклов
                if test2 == True and collide_index == 0:
                    break
                   
                if test2 == True:
                    curr_coord = local_collision_point(proj_point, surfaces, collide_index, CENTER_CIRCLE, CENTER_AXIS)
                    collide += 1
                    print('collide', collide_index, 'номер молекулы ->', n)
                    break
            
            # Критическая ошибка
            if collide_index == -1:
                print("Первая молекула исчезла")
                sys.exit(1)
                
            # Проверка на то, что молекула покинула канал и ее учет
            if collide_index == 1:
                outcome += 1
                break
            
            # Проверка на то, что молекула вернулась на вход и её учет
            if collide_index == 0:
                backflow += 1
                break
            
            # Вернем систему в исходное состояние
            # Задаем центр СК и обнуляем счетчик времени dt
            CENTER_AXIS = [0, 0, 0]
            thetta = 0
            dt = 0
        
            # Рассчитаем начальные точки поверхностей для того, чтобы найти новую стартовую точку и вектор
            CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
            points_array = get_surface_point(channel_height, thetta, points_array)
            
            # Рассчитываем коэф. поверхностей
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
            
            # Рассчитываем положение новой точки
            new_point = get_collision_point(collide_index, surfaces, proj_point, curr_coord)
            
            
            
            # Определимся, с какой поверхности мы вылетаем и запишем условия для разных случаев
            # Если молекула столкнулась с любой из лопаток
            if collide_index == 2 or collide_index == 4:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #генерация вектора
                vector = get_start_vector()
                
                #перевод вектора относительно другой плоскости (возможно здесь не правильно учтена нормаль)
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                
                #рассчет линейной скорости в точке столкновения
                blade_speed = get_blade_point_speed(new_point, speed_periphery, radius, radius_periphery, CENTER_CIRCLE)
                
                #сложение начального вектора с вектором скорости лопатки
                new_start_vector, new_speed = add_vectors(new_start_vector, molecule_speed, surfaces, collide_index, blade_speed)
                
                #сохранение данный о предыдущей и новой поверхностях старта 
                prev_index = current_index
                current_index = collide_index
                
                #новая стартовая точка
                start_point = new_point
                start_vector = new_start_vector
                molecule_speed = new_speed
                
                #переходим к следующему старту одной и той же молекулы
                continue
            
            # Если молекула попала в зазор
            if collide_index == 3 or collide_index == 5:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #перекидываем молекулу на другой зазор по свойству симметричности
                start_point = get_simmetric_point(new_point, surfaces, collide_index)
                if collide_index == 3:
                    prev_index = current_index
                    current_index = 5
                if collide_index == 5:
                    prev_index = current_index
                    current_index = 3
                    
                #переходим к следующему старту одной и той же молекулы
                continue
            
            # Если молекула столкнулась с другими поверхностями (ротор, расточка, верхушки лопаток)
            if collide_index == 6 or collide_index == 7 or collide_index == 8 or collide_index == 9:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #генерация вектора
                vector = get_start_vector()
                
                #перевод вектора относительно другой плоскости старта
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                
                #сохранение данный о предыдущей и новой поверхностях старта
                prev_index = current_index
                current_index = collide_index
                
                #новая стартовая точка
                start_point = new_point
                start_vector = new_start_vector
                
                #переходим к следующему старту одной и той же молекулы
                continue
    N_plus[spd] = outcome
    N_minus[spd] = backflow
    N_propal[spd] = small_count_collide
# Вывод результатов
print(N_plus)
print(N_minus)
print(N_propal)

[4.842536729654428, 0, 28.51930125517374]
[0.9753197634351882, 0.1579940765041906, -0.15423757921569048]
collide 4 номер молекулы -> 0
[-4.501876187440588, 0, 26.571881397498732]
[0.24547898161707343, 0.9317496438583527, 0.2675493800293272]
[-4.09174606990299, 0, 22.854033113626887]
[0.10985278859944421, 0.8879057130721355, 0.4467166994089076]
[-2.0091773935857953, 0, 23.227713811537924]
[-0.25051611259479944, 0.5462033262430818, -0.799314458602735]
[-3.3727495499319664, 0, 21.878638538458695]
[0.9222596236310832, 0.22793245889845198, -0.31222424761757106]
collide 4 номер молекулы -> 4
[3.054700003617067, 0, 28.824434455333385]
[0.04487817564177406, 0.998992305617352, 0.0005680390773722063]
collide 4 номер молекулы -> 5
[-3.8468376181038386, 0, 25.880168555864227]
[0.0657420254211359, 0.8786248523240698, 0.4729654902548739]
[2.943609940925926, 0, 8.434872436163964]
[-0.5818115334334582, 0.7580417731976791, 0.2947337945521799]
collide 4 номер молекулы -> 7
[0.5204053887231552, 0, 20.195

KeyboardInterrupt: 

In [126]:
# Переосмысление (обратный поток)

collide_index = -1


# Для записи результатов
N_plus=[0,0,0,0,0,0,0,0,0]
N_minus=[0,0,0,0,0,0,0,0,0]
N_propal=[0,0,0,0,0,0,0,0,0]
V2_Uh=[0,0,0,0,0,0,0,0,0]

# Исследование зависимости проводимости от величины межлопаточного зазора
for spd in range(9):
    if spd < 3:
        continue
    speed_periphery = Uh * 0.2 * spd
    speed_center = speed_periphery * radius / radius_periphery
    omega_center = speed_center / radius
    
    # Переменные для проверки и debug'a
    outcome = 0 # ушедшие на выход
    backflow = 0 # ушедшие обратно на вход
    collide = 0 # количество столкнувшихся
    small_count_collide = 0 # количество пропавших
    
    # Прогон молекул (в идеале 100K)
    for n in range (5000):

        # Индекс поверхности входа (0-прямой поток, 1-обратный поток)
        current_index = 1
        
        # Задаем центр СК (расположен в середине низа поверхности входа) и добавляем счетчик времени dt и угол поворота thetta
        CENTER_AXIS = [0, 0, 0]
        thetta = 0
        dt = 0
        
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_back_start_point()
        start_vector = get_back_start_vector()
        molecule_speed = get_molecule_speed(Uh)

        # Костыль для обновления точки в цикли по времени
        curr_point = start_point
        
        # Рассчитаем начальные точки поверхностей для их задания
        CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
        points_array = get_surface_point(channel_height, thetta, points_array)
        
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
    
        
        # Цикл определяющий количество доступных отражений для молекулы (пусть пока будет 5)
        a = 5 # Переменная для количества возможных столкновений
        for count_col in range(a):
            
            
            # Создаем цикл для изменения времени (сейчас dt=10^-9)
            for time in range(3544):
                
                dt = time * 0.0000000001
            
                # Перемещаем центр СК и рассчитываем новые поверхности
                CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
                points_array = get_surface_point(channel_height, thetta, points_array)
                for j in range(len(surfaces)):
                    surfaces = get_surfaces(points_array)
            
                # Смещаем молекулу (Запоминаем прошлое на всякий)
                prev_point = curr_point
                curr_point = get_new_point(prev_point, start_vector, dt, molecule_speed)
            
                # test1 - попала в плоскость, test2 - попала в поверхность
                test1 = False
                test2 = False
                
                # Проверяем наличие попадания молекулы в поверхности
                for index in range(len(surfaces)):
                    
                    # Условие различия стартовой и проверяемой поверхностей
                    if index == current_index:
                        continue
                        
                    
                    # Проверяем попадание в плоскость
                    test1, rastoyaniye = hardcheck(index, surfaces, curr_point)
                    #if index == 1:
                        #print("расстояние до", index, 'равно', rastoyaniye)
                    if test1 == False:
                        continue
                    
                    # Проверяем на принадлежность поверхности
                    proj_point = projection(index, surfaces, curr_point)
                    test2 = area(index, points_array, proj_point)
                    if test2 == False:
                        continue
                    
                        
                    # Если test2 = true, то точка столкнулась с поверхностью
                    if test2 == True:
                        collide_index = index
                        break
                    
                       
                # Если молекула пересекла диапазон по Y с прибавкой в 20%, то выходим с цикла, получаем ошибку
                if (curr_point[1] > total_width*1.5 or curr_point[1] < -0.5*total_width):
                    break
                        
                # Если молекула улетела на выход выходим из всех циклов (времени и столкновений)
                if test2 == True and collide_index == 1:
                    break
                
                # Если молекула улетела на вход выходим из всех циклов
                if test2 == True and collide_index == 0:
                    break
                   
                if test2 == True:
                    curr_coord = local_collision_point(proj_point, surfaces, collide_index, CENTER_CIRCLE, CENTER_AXIS)
                    collide += 1
                    print('collide', collide_index, 'номер молекулы ->', n)
                    break
            
            # Критическая ошибка
            if collide_index == -1:
                print("Первая молекула исчезла")
                sys.exit(1)
                
            # Проверка на то, что молекула покинула канал и ее учет
            if collide_index == 1:
                outcome += 1
                break
            
            # Проверка на то, что молекула вернулась на вход и её учет
            if collide_index == 0:
                backflow += 1
                break
            
            # Вернем систему в исходное состояние
            # Задаем центр СК и обнуляем счетчик времени dt
            CENTER_AXIS = [0, 0, 0]
            thetta = 0
            dt = 0
        
            # Рассчитаем начальные точки поверхностей для того, чтобы найти новую стартовую точку и вектор
            CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
            points_array = get_surface_point(channel_height, thetta, points_array)
            
            # Рассчитываем коэф. поверхностей
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
            
            # Рассчитываем положение новой точки
            new_point = get_collision_point(collide_index, surfaces, proj_point, curr_coord)
            
            
            
            # Определимся, с какой поверхности мы вылетаем и запишем условия для разных случаев
            # Если молекула столкнулась с любой из лопаток
            if collide_index == 2 or collide_index == 4:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #генерация вектора
                vector = get_start_vector()
                
                #перевод вектора относительно другой плоскости (возможно здесь не правильно учтена нормаль)
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                
                #рассчет линейной скорости в точке столкновения
                blade_speed = get_blade_point_speed(new_point, speed_periphery, radius, radius_periphery, CENTER_CIRCLE)
                
                #сложение начального вектора с вектором скорости лопатки
                new_start_vector, new_speed = add_vectors(new_start_vector, molecule_speed, surfaces, collide_index, blade_speed)
                
                #сохранение данный о предыдущей и новой поверхностях старта 
                prev_index = current_index
                current_index = collide_index
                
                #новая стартовая точка
                start_point = new_point
                start_vector = new_start_vector
                molecule_speed = new_speed
                
                #переходим к следующему старту одной и той же молекулы
                continue
            
            # Если молекула попала в зазор
            if collide_index == 3 or collide_index == 5:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #перекидываем молекулу на другой зазор по свойству симметричности
                start_point = get_simmetric_point(new_point, surfaces, collide_index)
                if collide_index == 3:
                    prev_index = current_index
                    current_index = 5
                if collide_index == 5:
                    prev_index = current_index
                    current_index = 3
                    
                #переходим к следующему старту одной и той же молекулы
                continue
            
            # Если молекула столкнулась с другими поверхностями (ротор, расточка, верхушки лопаток)
            if collide_index == 6 or collide_index == 7 or collide_index == 8 or collide_index == 9:
                
                #если молекула не успела вылететь за ограниченное количество столкновений
                if count_col == a-1:
                    small_count_collide += 1
                    break
                
                #генерация вектора
                vector = get_start_vector()
                
                #перевод вектора относительно другой плоскости старта
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                
                #сохранение данный о предыдущей и новой поверхностях старта
                prev_index = current_index
                current_index = collide_index
                
                #новая стартовая точка
                start_point = new_point
                start_vector = new_start_vector
                
                #переходим к следующему старту одной и той же молекулы
                continue
    N_plus[spd] = outcome
    N_minus[spd] = backflow
    N_propal[spd] = small_count_collide
# Вывод результатов
print(N_plus)
print(N_minus)
print(N_propal)

collide 2 номер молекулы -> 0
collide 2 номер молекулы -> 1
collide 6 номер молекулы -> 2
collide 2 номер молекулы -> 3
collide 2 номер молекулы -> 4
collide 2 номер молекулы -> 6
collide 2 номер молекулы -> 7
collide 2 номер молекулы -> 9
collide 4 номер молекулы -> 9
collide 2 номер молекулы -> 10
collide 2 номер молекулы -> 11
collide 2 номер молекулы -> 12
collide 2 номер молекулы -> 13
collide 6 номер молекулы -> 14
collide 2 номер молекулы -> 15
collide 2 номер молекулы -> 16
collide 2 номер молекулы -> 17
collide 2 номер молекулы -> 18
collide 2 номер молекулы -> 19
collide 2 номер молекулы -> 20
collide 2 номер молекулы -> 21
collide 2 номер молекулы -> 22
collide 2 номер молекулы -> 23
collide 2 номер молекулы -> 24
collide 6 номер молекулы -> 25
collide 2 номер молекулы -> 25
collide 2 номер молекулы -> 26
collide 4 номер молекулы -> 26
collide 2 номер молекулы -> 27
collide 2 номер молекулы -> 28
collide 2 номер молекулы -> 29
collide 3 номер молекулы -> 30
collide 6 номер м

In [ ]:
# Текущие тесты

problem1 = 0

# Тело программы для тестов
N_plus=[0,0,0,0,0,0,0,0,0]
N_minus=[0,0,0,0,0,0,0,0,0]
N_propal=[0,0,0,0,0,0,0,0,0]
V2_Uh=[0,0,0,0,0,0,0,0,0]


# Исследование зависимости проводимости от величины межлопаточного зазора
for spd in range(1):
    speed_periphery = Uh * 0.6
    speed_center = speed_periphery * radius / radius_periphery
    print(speed_center)
    omega_center = speed_center / radius
    print(omega_center)
    # Переменные для проверки и debug'a
    successful_outcome = 0 # ушедшие на выход
    collide = 0 # количество столкнувшихся с поверхностями
    lost = 0 # количество не столкнувшихся (суммарное количество проблемных)
    backflow = 0 # ушедшие обратно на вход

    # Сигнализации проблем
    problem = 0 
    prob_nevcond = 0
    small_count_collide = 0
    
    # Прогон молекул (в идеале 1М)
    for n in range (2):
        
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        #print(molecule_speed)
        # Индекс поверхности входа
        current_index = 0
        
        # Задаем центр СК и добавляем счетчик времени dt
        CENTER_AXIS = [0, 0, 0]
        thetta = 0
        dt = 0
        
        # Рассчитаем начальные точки поверхностей для их задания
        #CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
        points_array = get_surface_point(channel_height, thetta, points_array)
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
    
        
        
        # Цикл определяющий количество доступных отражений для молекулы (пусть пока будет 4)
        a = 4 # Переменная для количества возможных столкновений
        for count_col in range(a):
            
            # Массивы для записи расстояний от точек до плоскости
            distance_prev = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
            distance_curr = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
            # Переменные для хранения текущей и новой точек
            prev_point = start_point
            curr_point = start_point
            
            # Список поверхностей, которые уже не прошли проверку
            nevcondit = []
            nevcondit.append(current_index)
            #print(nevcondit)
            
            # Создаем цикл для изменения времени (сейчас dt=10^-7)
            for time in range(150):
                
                dt = time * 0.0000001
                dt2 = dt + 0.0000001
            
                # Перемещаем центр СК и рассчитываем новые поверхности
                CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
                points_array = get_surface_point(channel_height, thetta, points_array)
                for j in range(len(surfaces)):
                    surfaces = get_surfaces(points_array)
            
                # Смещаем молекулу (Сохраняя предыдущее положение)
                prev_point = get_new_point(start_point, start_vector, dt, molecule_speed)
                curr_point = get_new_point(start_point, start_vector, dt2, molecule_speed)
            
                # Переменная показывающая попала молекула в поверхность или нет
                tester = False
                
                #print(len(surfaces))
            
                # Проверяем наличие попадания молекулы в поверхности
                for index in range(len(surfaces)):
                    
                    # Задаем переменную сигнал, чтобы пропускать поверхности, которые не подошли нам
                    signal = False # False означает, что пропускать не нужно!
                    
                    # Условие различия стартовой и проверяемой поверхностей
                    # Условия отсекающие лишние поверхности
                    for zet in range(len(nevcondit)):  
                        if index == nevcondit[zet]:
                            signal = True #переходим к следующему index
                            #print(index, signal,  'номер молекулы ->', n)
                            break
                    
                    # Проверим, если сигнал True, то перейдем к следующему индексу
                    if signal == True:
                        #print(index, signal, 'переход к следующему индексу', 'номер молекулы ->', n)
                        continue
                
                    # Запись расстояние до плоскостей в массивы
                    distance_prev[index] = get_distance(prev_point, index, surfaces)
                    distance_curr[index] = get_distance(curr_point, index, surfaces)
                
                    # Ищем момент изменения знака величины расстояния от молекула до плоскости
                    if distance_prev[index] * distance_curr[index] < 0:
                        
                        # Находим точку на плоскости методом middle_projection
                        point_on_surface = middle_projection(index, surfaces, prev_point, distance_prev, curr_point, distance_curr)
                        tester = check(points_array, index, point_on_surface)
                        
                        
                        # Если тестер выдает True - значит молекула попала в поверхность, выходим из цикла, записываем нужные данные
                        if tester == True:
                            collide_index = index
                            curr_point = point_on_surface
                            break
                        
                        # Если тестер выдает False - значит нужно исключить плоскость из дальнейшего расчета
                        if tester == False:
                            # Эта запись должна добавить индекс поверхности в исключение, больше мы её не просчитываем
                            nevcondit.append(index)
                
                # Если все поверхности в исключении выходим и фиксируем проблему
                if len(nevcondit) == 10:
                    prob_nevcond += 1
                    #print('НИЧЕГО НЕ ЗАДЕЛА')
                    break
                        
                # Если молекула пересекла диапазон по Y с прибавкой в 20%, то выходим с цикла, получаем ошибку
                if (curr_point[1] > total_width*1.5 or curr_point[1] < -0.5*total_width) and tester == False:
                    problem += 1
                    #print('ДАЛЕКО УЛЕТЕЛА')
                    break
                        
                # Если молекула улетела на выход выходим из всех циклов (времени и столкновений)
                if tester == True and collide_index == 1:
                    print('pass', 'номер молекулы ->', n)
                    break
                
                # Если молекула улетела на вход выходим из всех циклов
                if tester == True and collide_index == 0:
                    print('back', 'номер молекулы ->', n)
                    break
                   
                if tester == True:
                    current_distance = local_collision_point(curr_point, surfaces, collide_index, CENTER_CIRCLE, CENTER_AXIS)
                    collide += 1
                    print('collide', collide_index, 'номер молекулы ->', n)
                    # Подумать какие данные нужно вытащить, прежде чем выйти из цикла времени (Далее нужно сделать ветвление)
                    break
            
            # Проверка на не работоспособность
            if collide_index == -10:
                problem1 += 1
                print('Tester ->', tester, 'collide_index ->', collide_index)
                break
                
            # Проверка на то, что молекула покинула канал и ее учет
            if collide_index == 1:
                successful_outcome += 1
                break
            
            # Проверка на то, что молекула вернулась на вход и её учет
            if collide_index == 0:
                backflow += 1
                break
            
            # Вернем систему в исходное состояние
            # Задаем центр СК и обнуляем счетчик времени dt
            CENTER_AXIS = [0, 0, 0]
            thetta = 0
            dt = 0
        
            # Рассчитаем начальные точки поверхностей для того, чтобы найти новую стартовую точку и вектор
            CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
            points_array = get_surface_point(channel_height, thetta, points_array)
            # Рассчитываем коэф. поверхностей
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
            
            # Рассчитываем положение новой точки
            new_point = get_collision_point(collide_index, surfaces, point_on_surface, current_distance)
            print(new_point)
            # Определимся, с какой поверхности мы вылетаем и запишем условия для разных случаев
            
            # Если молекула столкнулась с любой из лопаток
            if collide_index == 2 or collide_index == 4:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                print(new_start_vector)
                blade_speed = get_blade_point_speed(new_point, speed_periphery, radius, radius_periphery, CENTER_CIRCLE)
                start_vector, new_speed = add_vectors(new_start_vector, molecule_speed, surfaces, collide_index, blade_speed)
                print(start_vector)
                molecule_speed = new_speed
                prev_index = current_index
                current_index = collide_index
                start_point = new_point
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
            # Если молекула попала в зазор
            if collide_index == 3 or collide_index == 5:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                start_point = get_simmetric_point(new_point, surfaces, collide_index)
                if collide_index == 3:
                    prev_index = current_index
                    current_index = 5
                if collide_index == 5:
                    prev_index = current_index
                    current_index = 3
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
            # Если молекула столкнулась с другими поверхностями (ротор, расточка, верхушки лопаток)
            if collide_index == 6 or collide_index == 7 or collide_index == 8 or collide_index == 9:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                print(new_start_vector)
                prev_index = current_index
                current_index = collide_index
                start_point = new_point
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
    N_plus[spd] = successful_outcome
    N_minus[spd] = backflow
    N_propal[spd] = small_count_collide
    V2_Uh[spd] = float(speed_periphery / Uh)

# Вывод результатов

print(V2_Uh)
print(N_plus)
print(N_minus)
print(N_propal)

print(problem1)



175200.83647207852
2502.8690924582647
pass номер молекулы -> 0
pass номер молекулы -> 1
[0.6, 0, 0, 0, 0, 0, 0, 0, 0]
[2, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
0


In [ ]:
# Новая попытка

N_plus=[0,0,0,0,0,0,0,0,0]
N_minus=[0,0,0,0,0,0,0,0,0]
N_propal=[0,0,0,0,0,0,0,0,0]
V2_Uh=[0,0,0,0,0,0,0,0,0]


# Исследование зависимости проводимости от величины межлопаточного зазора
for spd in range(1):
    
    speed_periphery = Uh * 0.6
    speed_center = speed_periphery * radius / radius_periphery
    omega_center = speed_center / radius
    
    # Переменные для проверки и debug'a
    successful_outcome = 0 # ушедшие на выход
    collide = 0 # количество столкнувшихся с поверхностями
    lost = 0 # количество не столкнувшихся (суммарное количество проблемных)
    backflow = 0 # ушедшие обратно на вход
    
    
    # Прогон молекул (в идеале 1М)
    for n in range (5):
        
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        
        # Индекс поверхности входа
        current_index = 0
        
        # Задаем центр СК и добавляем счетчик времени dt
        CENTER_AXIS = [0, 0, 0]
        thetta = 0
        dt = 0
        
        # Рассчитаем начальные точки поверхностей для их задания
        points_array = get_surface_point(channel_height, thetta, points_array)
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
    
        
        
        # Цикл определяющий количество доступных отражений для молекулы (пусть пока будет 4)
        a = 4 # Переменная для количества возможных столкновений
        for count_col in range(a):
            
            # Массивы для записи расстояний от точек до плоскости
            distance_prev = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
            distance_curr = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
            # Переменные для хранения текущей и новой точек
            prev_point = start_point
            curr_point = start_point
            
            # Список поверхностей, которые уже не прошли проверку
            nevcondit = []
            nevcondit.append(current_index)
            #print(nevcondit)
            
            # Создаем цикл для изменения времени (сейчас dt=10^-7)
            for time in range(150):
                
                dt = time * 0.0000001
                dt2 = dt + 0.0000001
            
                # Перемещаем центр СК и рассчитываем новые поверхности
                CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
                points_array = get_surface_point(channel_height, thetta, points_array)
                for j in range(len(surfaces)):
                    surfaces = get_surfaces(points_array)
            
                # Смещаем молекулу (Сохраняя предыдущее положение)
                prev_point = get_new_point(start_point, start_vector, dt, molecule_speed)
                curr_point = get_new_point(start_point, start_vector, dt2, molecule_speed)
            
                # Переменная показывающая попала молекула в поверхность или нет
                tester = False
                
                #print(len(surfaces))
            
                # Проверяем наличие попадания молекулы в поверхности
                for index in range(len(surfaces)):
                    
                    # Задаем переменную сигнал, чтобы пропускать поверхности, которые не подошли нам
                    signal = False # False означает, что пропускать не нужно!
                    
                    # Условие различия стартовой и проверяемой поверхностей
                    # Условия отсекающие лишние поверхности
                    for zet in range(len(nevcondit)):  
                        if index == nevcondit[zet]:
                            signal = True #переходим к следующему index
                            #print(index, signal,  'номер молекулы ->', n)
                            break
                    
                    # Проверим, если сигнал True, то перейдем к следующему индексу
                    if signal == True:
                        #print(index, signal, 'переход к следующему индексу', 'номер молекулы ->', n)
                        continue
                
                    # Запись расстояние до плоскостей в массивы
                    distance_prev[index] = get_distance(prev_point, index, surfaces)
                    distance_curr[index] = get_distance(curr_point, index, surfaces)
                
                    # Ищем момент изменения знака величины расстояния от молекула до плоскости
                    if distance_prev[index] * distance_curr[index] < 0:
                        
                        # Находим точку на плоскости методом middle_projection
                        point_on_surface = middle_projection(index, surfaces, prev_point, distance_prev, curr_point, distance_curr)
                        tester = check(points_array, index, point_on_surface)
                        
                        
                        # Если тестер выдает True - значит молекула попала в поверхность, выходим из цикла, записываем нужные данные
                        if tester == True:
                            collide_index = index
                            curr_point = point_on_surface
                            break
                        
                        # Если тестер выдает False - значит нужно исключить плоскость из дальнейшего расчета
                        if tester == False:
                            # Эта запись должна добавить индекс поверхности в исключение, больше мы её не просчитываем
                            nevcondit.append(index)
                
                # Если все поверхности в исключении выходим и фиксируем проблему
                if len(nevcondit) == 10:
                    prob_nevcond += 1
                    #print('НИЧЕГО НЕ ЗАДЕЛА')
                    break
                        
                # Если молекула пересекла диапазон по Y с прибавкой в 20%, то выходим с цикла, получаем ошибку
                if (curr_point[1] > total_width*1.5 or curr_point[1] < -0.5*total_width) and tester == False:
                    problem += 1
                    #print('ДАЛЕКО УЛЕТЕЛА')
                    break
                        
                # Если молекула улетела на выход выходим из всех циклов (времени и столкновений)
                if tester == True and collide_index == 1:
                    print('pass', 'номер молекулы ->', n)
                    break
                
                # Если молекула улетела на вход выходим из всех циклов
                if tester == True and collide_index == 0:
                    print('back', 'номер молекулы ->', n)
                    break
                   
                if tester == True:
                    current_distance = local_collision_point(curr_point, surfaces, collide_index, CENTER_CIRCLE, CENTER_AXIS)
                    collide += 1
                    print('collide', collide_index, 'номер молекулы ->', n)
                    # Подумать какие данные нужно вытащить, прежде чем выйти из цикла времени (Далее нужно сделать ветвление)
                    break
            
            # Проверка на не работоспособность
            if collide_index == -10:
                problem1 += 1
                print('Tester ->', tester, 'collide_index ->', collide_index)
                break
                
            # Проверка на то, что молекула покинула канал и ее учет
            if collide_index == 1:
                successful_outcome += 1
                break
            
            # Проверка на то, что молекула вернулась на вход и её учет
            if collide_index == 0:
                backflow += 1
                break
            
            # Вернем систему в исходное состояние
            # Задаем центр СК и обнуляем счетчик времени dt
            CENTER_AXIS = [0, 0, 0]
            thetta = 0
            dt = 0
        
            # Рассчитаем начальные точки поверхностей для того, чтобы найти новую стартовую точку и вектор
            CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
            points_array = get_surface_point(channel_height, thetta, points_array)
            # Рассчитываем коэф. поверхностей
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
            
            # Рассчитываем положение новой точки
            new_point = get_collision_point(collide_index, surfaces, point_on_surface, current_distance)
            print(new_point)
            # Определимся, с какой поверхности мы вылетаем и запишем условия для разных случаев
            
            # Если молекула столкнулась с любой из лопаток
            if collide_index == 2 or collide_index == 4:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                print(new_start_vector)
                blade_speed = get_blade_point_speed(new_point, speed_periphery, radius, radius_periphery, CENTER_CIRCLE)
                start_vector, new_speed = add_vectors(new_start_vector, molecule_speed, surfaces, collide_index, blade_speed)
                print(start_vector)
                molecule_speed = new_speed
                prev_index = current_index
                current_index = collide_index
                start_point = new_point
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
            # Если молекула попала в зазор
            if collide_index == 3 or collide_index == 5:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                start_point = get_simmetric_point(new_point, surfaces, collide_index)
                if collide_index == 3:
                    prev_index = current_index
                    current_index = 5
                if collide_index == 5:
                    prev_index = current_index
                    current_index = 3
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
            # Если молекула столкнулась с другими поверхностями (ротор, расточка, верхушки лопаток)
            if collide_index == 6 or collide_index == 7 or collide_index == 8 or collide_index == 9:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                print(new_start_vector)
                prev_index = current_index
                current_index = collide_index
                start_point = new_point
                print('prev_index ->',prev_index, 'current_index ->', current_index,'следующая итерация')
                continue
            
    N_plus[spd] = successful_outcome
    N_minus[spd] = backflow
    N_propal[spd] = small_count_collide
    V2_Uh[spd] = float(speed_periphery / Uh)

# Вывод результатов

print(V2_Uh)
print(N_plus)
print(N_minus)
print(N_propal)

print(problem1)

In [35]:
# Текущие тесты обратного потока

# Тело программы для тестов

problem1 = 0

# Тело программы для тестов
N_plus=[0,0,0,0,0,0,0,0,0]
N_minus=[0,0,0,0,0,0,0,0,0]
N_propal=[0,0,0,0,0,0,0,0,0]
V2_Uh=[0,0,0,0,0,0,0,0,0]


# Исследование зависимости проводимости от величины межлопаточного зазора
for spd in range(1):
    speed_periphery = Uh * 0.2 * 0
    speed_center = speed_periphery * radius / radius_periphery
    # Переменные для проверки и debug'a
    successful_outcome = 0 # ушедшие на выход
    collide = 0 # количество столкнувшихся с поверхностями
    lost = 0 # количество не столкнувшихся (суммарное количество проблемных)
    backflow = 0 # ушедшие обратно на вход

    # Сигнализации проблем
    problem = 0 
    prob_nevcond = 0
    small_count_collide = 0
    
    # Прогон молекул (в идеале 1М)
    for n in range (1000):
        
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_back_start_point()
        start_vector = get_back_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        
        # Индекс поверхности входа
        current_index = 1
        
        # Задаем центр СК и добавляем счетчик времени dt
        CENTER_AXIS = [0, 0, 0]
        dt = 0
        
        # Рассчитаем начальные точки поверхностей для того, чтобы найти стартовые точку и вектор
        CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
        points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
    
        
        
        # Цикл определяющий количество доступных отражений для молекулы (пусть пока будет 4)
        a = 4 # Переменная для количества возможных столкновений
        for count_col in range(a):
            
            # Массивы для записи расстояний от точек до плоскости
            distance_prev = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
            distance_curr = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
            # Переменные для хранения текущей и новой точек
            prev_point = start_point
            curr_point = start_point
            
            # Список поверхностей, которые уже не прошли проверку
            nevcondit = []
            nevcondit.append(current_index)
            
            # Создаем цикл для изменения времени (сейчас dt=10^-8)
            for time in range(345):
                dt = time * 0.00000001
            
                # Перемещаем центр СК и рассчитываем новые поверхности
                CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
                points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
                for j in range(len(surfaces)):
                    surfaces = get_surfaces(points_array)
            
                # Смещаем молекулу (Сохраняя предыдущее положение)
                prev_point = curr_point
                curr_point = get_new_point(start_point, start_vector, dt, molecule_speed)
            
                # Переменная показывающая попала молекула в поверхность или нет
                tester = False
            
                # Проверяем наличие попадания молекулы в поверхности
                for index in range(len(surfaces)):
                    
                    # Задаем переменную сигнал, чтобы пропускать поверхности, которые не подошли нам
                    signal = False # False означает, что пропускать не нужно!
                    
                    # Условие различия стартовой и проверяемой поверхностей
                    # Условия отсекающие лишние поверхности
                    for zet in range(len(nevcondit)):  
                        if index == nevcondit[zet]:
                            signal = True #переходим к следующему index
                            break
                    
                    # Проверим, если сигнал True, то перейдем к следующему индексу
                    if signal == True:
                        continue
                
                    # Запись расстояние до плоскостей в массивы
                    distance_prev[index] = distance_curr[index]
                    distance_curr[index] = get_distance(curr_point, index, surfaces)
                    
                    # Ищем момент изменения знака величины расстояния от молекула до плоскости
                    if distance_prev[index] * distance_curr[index] < 0:
                        
                        # Находим точку на плоскости методом middle_projection
                        point_on_surface = middle_projection(index, surfaces, prev_point, distance_prev, curr_point, distance_curr)
                        tester = check(points_array, index, point_on_surface)
                        
                        
                        # Если тестер выдает True - значит молекула попала в поверхность, выходим из цикла, записываем нужные данные
                        if tester == True:
                            collide_index = index
                            curr_point = point_on_surface
                            break
                        
                        # Если тестер выдает False - значит нужно исключить плоскость из дальнейшего расчета
                        if tester == False:
                            # Эта запись должна добавить индекс поверхности в исключение, больше мы её не просчитываем
                            nevcondit.append(index)
                
                # Если все поверхности в исключении выходим и фиксируем проблему
                if len(nevcondit) == 10:
                    prob_nevcond += 1
                    break
                        
                # Если молекула пересекла диапазон по Y с прибавкой в 20%, то выходим с цикла, получаем ошибку
                if curr_point[1] > (points_array[1][1][1])*1.2 or curr_point[1] < -0.5*(points_array[1][1][1]):
                    problem += 1
                    break
                        
                # Если молекула улетела на вход выходим из всех циклов (времени и столкновений)
                if tester == True and collide_index == 0:
                    break
                
                # Если молекула улетела на выход выходим из всех циклов
                if tester == True and collide_index == 1:
                    break
                   
                if tester == True:
                    current_distance = local_collision_point(curr_point, surfaces, collide_index, CENTER_CIRCLE, CENTER_AXIS)
                    collide += 1
                    print('collide', collide_index, 'номер молекулы ->', n)
                    # Подумать какие данные нужно вытащить, прежде чем выйти из цикла времени (Далее нужно сделать ветвление)
                    break
            
            # Проверка на не работоспособность
            if collide_index == -1:
                problem1 += 1
                break
                
            # Проверка на то, что молекула покинула канал и ее учет
            if collide_index == 0:
                successful_outcome += 1
                break
            
            # Проверка на то, что молекула вернулась на вход и её учет
            if collide_index == 1:
                backflow += 1
                break
            
            # Вернем систему в исходное состояние
            # Задаем центр СК и обнуляем счетчик времени dt
            CENTER_AXIS = [0, 0, 0]
            dt = 0
        
            # Рассчитаем начальные точки поверхностей для того, чтобы найти новую стартовую точку и вектор
            CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
            points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
            # Рассчитываем коэф. поверхностей
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
            
            # Рассчитываем положение новой точки
            new_point = get_collision_point(collide_index, surfaces, point_on_surface, current_distance)
            
            # Определимся, с какой поверхности мы вылетаем и запишем условия для разных случаев
            
            # Если молекула столкнулась с любой из лопаток
            if collide_index == 2 or collide_index == 4:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                new_start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                blade_speed = get_blade_point_speed(new_point, speed_periphery, radius, radius_periphery, CENTER_CIRCLE)
                start_vector, new_speed = add_vectors(new_start_vector, molecule_speed, surfaces, collide_index, blade_speed)
                molecule_speed = new_speed
                current_index = collide_index
                start_point = new_point
                continue
            
            # Если молекула попала в зазор
            if collide_index == 3 or collide_index == 5:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                start_point = get_simmetric_point(new_point, surfaces, collide_index)
                if collide_index == 3:
                    current_index = 5
                if collide_index == 5:
                    current_index = 3
                continue
            
            # Если молекула столкнулась с другими поверхностями (ротор, расточка, верхушки лопаток)
            if collide_index == 6 or collide_index == 7 or collide_index == 8 or collide_index == 9:
                if count_col == a-1:
                    small_count_collide += 1
                    break
                vector = get_start_vector()
                start_vector = get_new_start_vector(points_array, vector, surfaces, collide_index)
                current_index = collide_index
                start_point = new_point
                continue
            
    N_plus[spd] = successful_outcome
    N_minus[spd] = backflow
    N_propal[spd] = small_count_collide
    V2_Uh[spd] = float(speed_periphery / Uh)

# Вывод результатов

print(V2_Uh)
print(N_plus)
print(N_minus)
print(N_propal)

print(problem1)

#print('successful_outcome', successful_outcome)
#print('backflow', backflow)
#print('collide', collide)
#print('lost', lost)

#print('problem', problem)
#print('prob_nevcond', prob_nevcond)
#print('small_count_collide', small_count_collide)
#print('problem1', problem1)


collide 4 номер молекулы -> 0


TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
# (BACKUP) Тело программы

# Переменные для вывода
successful_outcome = 0
problem = 0

# Исследование зависимости проводимости от величины межлопаточного зазора
for channel_height in [1]:
    # Прогон молекул (в идеале 1М)
    for n in range (100):
        
        # Задаем центр СК и добавляем счетчик времени dt
        CENTER_AXIS = [0, 0, 0]
        dt = 0
        
        # Рассчитаем начальные точки поверхностей для того, чтобы найти стартовые точку и вектор
        CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
        points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
        # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        current_index = 0
        
        # Создаем цикл для изменения времени (сейчас dt=10^-7)
        for time in range(1500):
            dt = time * 0.0001
            
            # Перемещаем центр СК и рассчитываем новые поверхности
            CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
            points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
             
            # Смещаем молекулу 
            current_point = get_new_point(start_point, start_vector, dt, molecule_speed)
            
            tester = False
            # Проверяем наличие попадания молекулы в поверхности
            for index in range(len(surfaces)):
                # Условие различия стартовой поверхности и проверяемой
                if index == current_index:
                    continue #переходим к следующему index
                
                # Непосредственно сама проверка
                tester = check_condition(points_array, current_point, index, surfaces)
                #print(tester, index)
                if tester == True:
                    collide += 1
                    new_index = index
                    break
                if tester == False and index == len(surfaces):
                    problem += 1
                    break
        
        # Обновление цикла
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        current_index = 0

# Вывод результатов
print(successful_outcome)

In [ ]:
# (BACKUP) Тело программы без цикла для количества столкновений

# Переменные для проверки и debug'a
successful_outcome = 0
problem = 0
collide = 0
lost = 0
backflow = 0

# Исследование зависимости проводимости от величины межлопаточного зазора
for channel_height in [1]:
    
    # Находим стартовую точку молекулы, а также направляющий вектор и скорость молекулы
    start_point = get_start_point()
    start_vector = get_start_vector()
    molecule_speed = get_molecule_speed(Uh)
    print(start_vector)
    # Индекс поверхности входа
    current_index = 0
    
    # Прогон молекул (в идеале 1М)
    for n in range (5000):
        
        # Задаем центр СК и добавляем счетчик времени dt
        CENTER_AXIS = [0, 0, 0]
        dt = 0
        # Рассчитаем начальные точки поверхностей для того, чтобы найти стартовые точку и вектор
        CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
        points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
        # Рассчитываем коэф. поверхностей
        for j in range(len(surfaces)):
            surfaces = get_surfaces(points_array)
            
        # Список поверхностей, которые уже не прошли проверку
        nevcondit = []
        nevcondit.append(current_index)
        
        # Массивы для записи расстояний от точек до плоскости
        distance_prev = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
        distance_curr = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
        
        # Переменные для хранения текущей и новой точек
        prev_point = start_point
        curr_point = start_point
        
        # Создаем цикл для изменения времени (сейчас dt=10^-7)
        for time in range(1200):
            dt = time * 0.0000001
            
            # Перемещаем центр СК и рассчитываем новые поверхности
            CENTER_AXIS, tangent_angle = center(CENTER_AXIS, dt, speed_center)
            points_array = get_surf_point(channel_height, CENTER_AXIS, tangent_angle, points_array)
            for j in range(len(surfaces)):
                surfaces = get_surfaces(points_array)
             
            # Смещаем молекулу (Сохраняя предыдущее положение)
            prev_point = curr_point
            curr_point = get_new_point(start_point, start_vector, dt, molecule_speed)
            
            # Переменная показывающая попала молекула в поверхность или нет
            tester = False
            
            # Проверяем наличие попадания молекулы в поверхности
            for index in range(len(surfaces)):
                
                # Условие различия стартовой и проверяемой поверхностей
                # Условия отсекающие лишние поверхности
                for zet in range(len(nevcondit)):  
                    if index == nevcondit[zet]:
                        continue #переходим к следующему index
                
                # Запись расстояние до плоскостей в массивы
                distance_prev[index] = distance_curr[index]
                distance_curr[index] = get_distance(curr_point, index, surfaces)
                
                # Ищем момент изменения знака величины расстояния от молекула до плоскости
                if distance_prev[index] * distance_curr[index] < 0:
                    # Находим точку на плоскости методом middle_projection
                    point_on_surface = middle_projection(index, surfaces, prev_point, distance_prev, curr_point, distance_curr)
                    tester = check(points_array, index, point_on_surface)
                    # Если тестер выдает False - значит нужно исключить плоскость из дальнейшего расчета
                    # Если тестер выдает True - значит молекула попала в поверхность, выходим из цикла, записываем нужные данные
                    if tester == True:
                        col_index = index
                        
                        break
                    if tester == False:
                        # Эта запись должна добавить индекс поверхности в исключение, больше мы её не просчитываем
                        nevcondit.append(index)
                        
            # Если молекула пересекла диапазон по Y, то выходим с цикла, получаем ошибку
            if curr_point[1] > (points_array[1][1][1])*1.15:
                problem += 1
                #print(curr_point)
                #print(points_array)
                break
                
                # Если проверка не пройдена по истечении времени, то выходим из цикла (можно добавить выход по координате Y, т.к. не больше канала)
                #if tester == False and index == len(surfaces):
                    #break
                    
            if tester == True:
                collide += 1
                print('collide', col_index, 'номер молекулы ->', n)
                # Подумать какие данные нужно вытащить, прежде чем выйти из цикла времени (Здесь по идее далее нужно ветвление)
                break
        
        # Обновление цикла
        start_point = get_start_point()
        start_vector = get_start_vector()
        molecule_speed = get_molecule_speed(Uh)
        current_index = 0

# Вывод результатов
#print(successful_outcome)
print(problem)
print(collide)

In [83]:
# Тест движения центр и остального
CENTER_AXIS = [0, 0, 0]
thetta = 0
dt = 0
CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
points_array = get_surface_point(channel_height, thetta, points_array)
for j in range(len(surfaces)):
        surfaces = get_surfaces(points_array)
        #print(surfaces[j])
print(surfaces)
print(points_array[0])
print(points_array[1])
print(CENTER_AXIS)
#print(Uh)
#print(blade_interval)
#print(blade_width)     
for i in range(3544):
    dt = i * 0.0000000001
    CENTER_AXIS, thetta = turning_center(CENTER_AXIS, dt, omega_center, thetta, radius)
    points_array = get_surface_point(channel_height, thetta, points_array)
    #print(points_array[0])
    for j in range(len(surfaces)):
        surfaces = get_surfaces(points_array)
        #print(surfaces[j])
print(surfaces)
print(points_array[0])
print(points_array[1])
print(CENTER_AXIS)

[(0.0, -1.0, 0.0, 0.0), (0.0, -1.0, -0.0, 3.0), (0.4996913621679091, 0.8654908273781193, 0.0351307600133839, 1.5004526865782988), (0.4999999990820982, 0.8660254021945832, -6.0593829481171725e-05, 2.813379787524617), (0.49969136216790905, 0.8654908273781193, -0.0351307600133839, -1.5004526865782986), (0.4999999990820961, 0.8660254021945843, 6.059382948043132e-05, -2.8133797875245907), (0.0, 0.0, 1.0, -31.143804500273188), (0.0, 0.0, 1.0, 0.0), (-2.8359888540733107e-14, -3.663152269844693e-14, 1.0, -30.14380450027336), (2.8359888540733107e-14, 4.3721494833630205e-14, 1.0, -30.14380450027336)]
[[-3.002758903152919, 0.0, 0.0], [3.002758903152919, 0.0, 0.0], [5.122016592929867, 0.0, 30.14380450027319], [-5.122016592929867, 0.0, 30.14380450027319], [5.62298534061338, 0.0, 31.14380450027319], [-5.62298534061338, 0.0, 31.14380450027319], [-5.623106528272563, 0.0, 30.143804500273177], [5.623106528272563, 0.0, 30.143804500273177]]
[[-8.198911325859552, 3.0, 0.0], [-2.1933935195537138, 3.0, 0.0],